- Instead of maintaining two separate classes, MultiHeadAttentionWrapper and CausalAttention, we can combine both of these concepts into a single MultiHeadAttention class.

- Also, in addition to just merging the MultiHeadAttentionWrapper with the CausalAttention code, we will make some other modifications to implement multi-head attention more efficiently.

- In the MultiHeadAttentionWrapper, multiple heads are implemented by creating a list of CausalAttention objects (self.heads), each representing a separate attention head.

- The CausalAttention class independently performs the attention mechanism, and the results from each head are concatenated.

- In contrast, the following MultiHeadAttention class integrates the multi-head functionality within a single class.

- It splits the input into multiple heads by reshaping the projected query, key, and value tensors and then combines the results from these heads after computing attention.

Let's take a look at the MultiHeadAttention class before we discuss it further:

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()

    # Ensure total output dimension can be evenly split across heads
    assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

    # Store total output dimension (after concatenating all heads)
    self.d_out = d_out

    # Number of attention heads
    self.num_heads = num_heads

    # Dimension handled by each head
    self.head_dim = d_out // num_heads

    # Linear projection to generate Query vectors
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)

    # Linear projection to generate Key vectors
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)

    # Linear projection to generate Value vectors
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    # Final linear layer applied after concatenating all heads
    self.out_proj = nn.Linear(d_out, d_out)

    # Register causal mask (upper triangular matrix)
    # This prevents tokens from attending to future tokens
    self.register_buffer(
        "mask",
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )

  def forward(self, x):
    # x shape = (batch_size, sequence_length, input_dimension)
    b, num_tokens, d_in = x.shape

    # Project input into Key space
    # Shape -> (batch, num_tokens, d_out)
    keys = self.W_key(x)

    # Project input into Query space
    queries = self.W_query(x)

    # Project input into Value space
    values = self.W_value(x)

    # Reshape to split into multiple heads
    # New shape -> (batch, num_tokens, num_heads, head_dim)
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

    # Move num_heads dimension before sequence dimension
    # Shape -> (batch, num_heads, num_tokens, head_dim)
    keys = keys.transpose(1, 2)
    queries = queries.transpose(1, 2)
    values = values.transpose(1, 2)

    # Compute attention scores using dot product
    # (batch, heads, tokens, head_dim) @ (batch, heads, head_dim, tokens)
    # Result -> (batch, heads, tokens, tokens)
    attn_scores = keys @ queries.transpose(2, 3)

    # Convert mask to boolean and slice according to current sequence length
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

    # Apply causal mask: future positions get -inf so softmax -> 0
    attn_scores.masked_fill_(mask_bool, -torch.inf)

    # Scale attention scores to stabilize gradients
    # Then apply softmax across last dimension (over tokens)
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5,
        dim=-1
    )

    # Multiply attention weights with values to get context
    # Shape -> (batch, heads, tokens, head_dim)
    context_vectors = (attn_weights @ values).transpose(1, 2)

    # Merge all heads back together
    # Shape -> (batch, tokens, d_out)
    context_vectors = context_vectors.contiguous().view(b, num_tokens, self.d_out)

    # Final projection layer
    context_vectors = self.out_proj(context_vectors)

    # Return final contextualized representations
    return context_vectors

In [ ]:
torch.manual_seed(123)

# Define the tensor with 3 rows and 6 columns
inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],  # Row 1
     [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],  # Row 2
     [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]  # Row 3
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

batch_size, context_length, d_in = batch.shape
d_out = 6
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)